In [1]:
import pandas
df =pd.read_csv('Detailed_Polling_Data.csv')

/Users/karthickkumarasamy/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


<IPython.core.display.Javascript object>

In [2]:
df.columns

Index(['Serial No. Of Polling Station', 'Dravida Munnetra Kazhagam',
       'Naam Tamilar Katchi', 'All India Anna Dravida Munnetra Kazhagam',
       'Puthiya Makkal Tamil Desam Katchi', 'Tamizhaga Vaazhvurimai Katchi',
       'Naam Indiar Party', 'Tamilaga Vettri Kazhagam', 'Independent',
       'Independent.1', 'Independent.2', 'Independent.3', 'Independent.4',
       'Independent.5', 'Independent.6', 'Independent.7', 'Independent.8',
       'Independent.9', 'Total of Valid Votes', 'No. Of Rejected Votes',
       'NOTA', 'Total', 'No. Of Tendered Votes',
       'Location and Name of Building in Which Polling Station Located',
       'Polling Areas', 'Category',
       'Building in Which Polling Station Located', 'location', 'Winner_Party',
       'Winner_Votes', 'Runner_Up_Votes', 'Margin_Of_Victory',
       'Runner_Up_Party', 'Dravida Munnetra Kazhagam_Rank',
       'All India Anna Dravida Munnetra Kazhagam_Rank',
       'Naam Tamilar Katchi_Rank', 'Puthiya Makkal Tamil Desam Katchi

In [4]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Load the new dataset (Update the filename to match your file)
df4 = pd.read_csv("Detailed_Polling_Data.csv")

# 2. Select the core major political forces based on your columns
core_parties = [
    'Dravida Munnetra Kazhagam', 
    'All India Anna Dravida Munnetra Kazhagam', 
    'Naam Tamilar Katchi', 
    'Tamilaga Vettri Kazhagam'
]

# Ensure zero issues with missing numbers
df4[core_parties] = df4[core_parties].fillna(0)

# 3. Calculate true total votes for normalization (Core + Independents + NOTA)
df4['Total_Calculated_Votes'] = df4[core_parties].sum(axis=1) + df4['Total_Independent_Votes'].fillna(0) + df4['NOTA'].fillna(0)

# Filter out empty entries to completely avoid mathematical errors
df4 = df4[df4['Total_Calculated_Votes'] > 0].copy()

# 4. Feature Engineering: Create normalized percentage shares
share_cols = []
for party in core_parties:
    col_name = f'{party}_share_pct'
    df4[col_name] = (df4[party] / df4['Total_Calculated_Votes']) * 100
    share_cols.append(col_name)

# Append strategic structural dimensions
df4['independent_share_pct'] = (df4['Total_Independent_Votes'].fillna(0) / df4['Total_Calculated_Votes']) * 100
feature_cols = share_cols + ['independent_share_pct', 'Margin_Percentage']

# Drop or fill edge-case missing numbers inside target features
df4[feature_cols] = df4[feature_cols].fillna(0)

# 5. Extract and Scale features for the ML model
X = df4[feature_cols]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Apply K-Means Clustering to group booths into 4 core segments
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df4['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# 7. Print the Profile Breakdown to help map the text identities
print("\n--- DATASET 4: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
profile = df4.groupby('Cluster_ID')[feature_cols].mean()
print(profile.round(2))

print("\n--- DATASET 4: BOOTH COUNT PER CLUSTER ---")
print(df4['Cluster_ID'].value_counts())

# 8. Export individual tactical files for campaign ground teams
for cluster_num in range(optimal_k):
    cluster_df = df4[df4['Cluster_ID'] == cluster_num][
        [
            'Serial No. Of Polling Station', 
            'Location and Name of Building in Which Polling Station Located', 
            'Polling Areas', 
            'Winner_Party', 
            'Margin_Percentage'
        ]
    ]
    filename = f"Dataset_4_Cluster_{cluster_num}_Booths.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign tracking files exported successfully.")



--- DATASET 4: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---
            Dravida Munnetra Kazhagam_share_pct  \
Cluster_ID                                        
0                                         30.53   
1                                         80.19   
2                                         29.45   
3                                         39.25   

            All India Anna Dravida Munnetra Kazhagam_share_pct  \
Cluster_ID                                                       
0                                                       13.03    
1                                                        1.70    
2                                                       17.87    
3                                                        9.87    

            Naam Tamilar Katchi_share_pct  Tamilaga Vettri Kazhagam_share_pct  \
Cluster_ID                                                                      
0                                    5.49                               4